# RAG Multi-Agent System - Debug & Testing Notebook

## Purpose
This notebook allows you to test and debug each component of the RAG multi-agent system individually:

1. **Schema Understanding** - See how Unity Catalog tables are being read and understood
2. **Column Selection Logic** - Test semantic column matching and selection
3. **RAG Integration** - Test document retrieval and context enrichment
4. **Query Decomposition** - See how complex questions are broken down
5. **Query Planning** - Test query generation for Genie
6. **Genie Execution** - Test actual query execution
7. **Result Validation** - Check if Genie responses match the question
8. **Follow-up Generation** - Test analytical follow-up questions
9. **End-to-End Flow** - Complete system test

---

## Setup & Configuration

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
import json
from pprint import pprint

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Python path updated")

In [ ]:
# Load environment variables
from dotenv import load_dotenv

env_path = project_root / ".env"
load_dotenv(env_path)

print(f"✓ Environment loaded from: {env_path}")
print(f"✓ Azure OpenAI Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT', 'NOT SET')}")
print(f"✓ Databricks Host: {os.getenv('DATABRICKS_HOST', 'NOT SET')}")
print(f"✓ RAG Enabled: {os.getenv('RAG_ENABLED', 'false')}")

In [ ]:
# Initialize core components
from src.core.config import config
from src.utils.llm import get_llm
from src.utils.logging import get_logger

logger = get_logger("debug_notebook")
llm = get_llm()

print("✓ Configuration loaded")
print(f"✓ LLM initialized: {config.azure_openai.gpt4o_deployment}")
print(f"✓ Unity Catalog: {config.databricks.unity_catalog}.{config.databricks.unity_schema}")
print(f"✓ Tables configured: {len(config.databricks.unity_tables)}")

---

## Section 1: Schema Understanding

**Purpose:** See exactly how Unity Catalog tables are being read and what information is available.

**What to check:**
- Are all tables being loaded?
- Are column names and types correct?
- Are column descriptions/comments available?
- Is there enough metadata for semantic matching?

In [ ]:
print("=" * 80)
print("SECTION 1: SCHEMA UNDERSTANDING")
print("=" * 80)
print()

In [ ]:
# Initialize Unity Schema Reader
from databricks.sdk import WorkspaceClient
from src.agent_simple import UnitySchemaReader

workspace_client = WorkspaceClient(
    host=config.databricks.host,
    token=config.databricks.token
)

schema_reader = UnitySchemaReader(workspace_client)

print("✓ Unity Schema Reader initialized")
print(f"✓ Reading schemas from: {config.databricks.unity_catalog}.{config.databricks.unity_schema}")
print(f"✓ Tables to read: {config.databricks.unity_tables}")

In [ ]:
# Read all schemas
schema_info = schema_reader.read_all_configured_schemas()

print("\n" + "=" * 80)
print("SCHEMA INFORMATION (As seen by LLM)")
print("=" * 80)
print(schema_info)
print("\n" + "=" * 80)

In [ ]:
# Parse and display structured schema info
print("\n📊 STRUCTURED SCHEMA ANALYSIS")
print("=" * 80)

for table_name in config.databricks.unity_tables:
    full_table_name = f"{config.databricks.unity_catalog}.{config.databricks.unity_schema}.{table_name}"
    print(f"\n🔍 Table: {full_table_name}")
    print("-" * 80)
    
    try:
        schema = schema_reader.read_table_schema(full_table_name)
        
        print(f"  Type: {schema.get('table_type', 'UNKNOWN')}")
        print(f"  Description: {schema.get('table_comment', 'No description')}")
        print(f"  Columns: {len(schema.get('columns', []))}")
        print()
        
        # Show columns with details
        print("  Column Details:")
        for col in schema.get('columns', []):
            col_name = col.get('name', 'unknown')
            col_type = col.get('type', 'unknown')
            col_comment = col.get('comment', 'No description')
            col_nullable = col.get('nullable', True)
            
            nullable_str = "nullable" if col_nullable else "NOT NULL"
            print(f"    • {col_name:30s} {col_type:15s} ({nullable_str})")
            if col_comment:
                print(f"      → {col_comment}")
        
    except Exception as e:
        print(f"  ❌ Error reading schema: {e}")

print("\n" + "=" * 80)

### ✍️ Section 1 Observations

**Questions to answer:**
1. Are all expected tables visible?
2. Do column descriptions provide enough context?
3. Are there any missing or unclear column types?
4. Would you add more descriptive comments to any columns?

**Your notes:**
- 
- 

---

## Section 2: Column Selection Logic

**Purpose:** Test how the system semantically matches user intent to table columns.

**What to test:**
- Does "sentiment" correctly map to sentiment-related columns?
- Does "location" correctly identify city/region columns?
- Does the system understand calculated metrics (conversion rate, etc.)?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 2: COLUMN SELECTION LOGIC")
print("=" * 80)
print()

In [ ]:
# Test schema analysis with various user questions
from langchain_core.messages import HumanMessage, AIMessage

test_questions = [
    "Show me sentiment for bangalore",
    "What's the conversion rate last month?",
    "Show me sales by region",
    "What's the average order value?",
    "Show customer satisfaction scores"
]

print("🧪 Testing Schema Analysis with various questions...\n")

In [ ]:
# Create schema analysis agent
from src.agent_simple import create_schema_analysis_agent

schema_agent = create_schema_analysis_agent(schema_reader, llm)

print("✓ Schema Analysis Agent created")

In [ ]:
# Test each question
for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"TEST {i}: {question}")
    print("=" * 80)
    
    # Create test state
    test_state = {
        "messages": [HumanMessage(content=question)],
        "schema_info": schema_info,
        "rag_context": "",
        "iterations": 0,
        "original_question": question
    }
    
    # Run analysis
    result = schema_agent(test_state)
    
    # Display results
    print(f"\n📊 ANALYSIS RESULT:\n")
    print(f"  Answerable: {result.get('is_answerable', 'Unknown')}")
    print(f"  Needs Clarification: {result.get('needs_clarification', False)}")
    print(f"  Next Agent: {result.get('next_agent', 'unknown')}")
    
    # Show AI response
    if result.get('messages'):
        for msg in result['messages']:
            if isinstance(msg, AIMessage):
                print(f"\n💬 AI Analysis:\n")
                print(msg.content)
    
    print(f"\n{'-' * 80}")

### ✍️ Section 2 Observations

**Questions to answer:**
1. Did the system correctly identify relevant columns for each question?
2. Were any questions marked as "needs clarification" when they shouldn't be?
3. Did the system make any incorrect assumptions?
4. Are the column selections logical and complete?

**Your notes:**
- 
- 

---

## Section 3: RAG Integration

**Purpose:** Test document retrieval and see how RAG context enriches queries.

**What to test:**
- Does RAG find relevant documents?
- Are similarity scores meaningful?
- Does RAG context help with query planning?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 3: RAG INTEGRATION")
print("=" * 80)
print()

In [ ]:
# Check RAG configuration
print("📋 RAG Configuration:")
print(f"  Enabled: {config.rag.enabled}")
print(f"  Documents Path: {config.rag.documents_path}")
print(f"  Vector Store Path: {config.rag.vector_store_path}")
print(f"  Top K: {config.rag.top_k}")
print(f"  Min Similarity: {config.rag.min_similarity}")
print(f"  Standalone QA: {config.rag.enable_standalone_qa}")
print(f"  Standalone Threshold: {config.rag.standalone_threshold}")

In [ ]:
# Initialize RAG store (if enabled)
if config.rag.enabled:
    from src.services.rag_store import RAGStore
    from src.utils.embeddings import get_embedding_service
    
    embeddings_service = get_embedding_service()
    rag_store = RAGStore(
        embedding_service=embeddings_service,
        vector_store_path=config.rag.vector_store_path,
        chunk_size=config.rag.chunk_size,
        chunk_overlap=config.rag.chunk_overlap
    )
    
    print("\n✓ RAG Store initialized")
    
    # Check if documents exist
    docs_path = Path(config.rag.documents_path)
    if docs_path.exists():
        doc_files = list(docs_path.glob("**/*.*"))
        print(f"✓ Documents path exists: {docs_path}")
        print(f"✓ Found {len(doc_files)} files")
        for doc_file in doc_files:
            print(f"    • {doc_file.name} ({doc_file.stat().st_size} bytes)")
    else:
        print(f"⚠️  Documents path does not exist: {docs_path}")
        print(f"    Create this directory and add documents to enable RAG.")
else:
    print("\n⚠️  RAG is disabled in configuration")
    print("    Set RAG_ENABLED=true in .env to enable RAG")
    rag_store = None

In [ ]:
# Test RAG search with sample queries
if rag_store:
    test_queries = [
        "What is sentiment score?",
        "How do we calculate conversion rate?",
        "What tables contain customer data?",
        "Explain NPS score calculation"
    ]
    
    print("\n🧪 Testing RAG Search...\n")
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n{'=' * 80}")
        print(f"QUERY {i}: {query}")
        print("=" * 80)
        
        try:
            results = rag_store.search(
                query=query,
                top_k=config.rag.top_k,
                min_similarity=config.rag.min_similarity
            )
            
            if results:
                print(f"\n✓ Found {len(results)} relevant documents:\n")
                for j, doc in enumerate(results, 1):
                    similarity = doc.metadata.get('similarity', 0)
                    filename = doc.metadata.get('filename', 'Unknown')
                    print(f"  {j}. [{filename}] (Similarity: {similarity:.3f})")
                    print(f"     Content: {doc.content[:200]}...")
                    print()
            else:
                print("\n✗ No relevant documents found")
                
        except Exception as e:
            print(f"\n❌ Error: {e}")
else:
    print("\n⚠️  Skipping RAG search tests (RAG not enabled)")

### ✍️ Section 3 Observations

**Questions to answer:**
1. Are documents being retrieved correctly?
2. Are similarity scores reasonable?
3. Would adding more documentation help?
4. What types of documents should be added?

**Your notes:**
- 
- 

**📁 To add RAG documents:**
1. Create directory: `./data/documents/`
2. Add files: `.md`, `.txt`, `.pdf`, `.docx`, `.json`, `.csv`
3. Examples:
   - `data_dictionary.md` - Column definitions
   - `metric_calculations.md` - Business logic formulas
   - `table_descriptions.md` - Table relationships
   - `business_glossary.json` - Term definitions

---

## Section 4: Query Decomposition

**Purpose:** Test how complex questions are broken down into simpler sub-questions.

**What to test:**
- Does the system break down complex questions appropriately?
- Are sub-questions logical and answerable?
- Do sub-questions cover all aspects of the original question?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 4: QUERY DECOMPOSITION")
print("=" * 80)
print()

In [ ]:
# Test query decomposition with complex questions
complex_questions = [
    "Show me sentiment analysis by city and compare with sales performance",
    "What's the correlation between customer satisfaction and conversion rates across different regions?",
    "Analyze year-over-year growth in revenue and identify top performing products by category",
    "Compare sentiment scores before and after product launch in bangalore and delhi"
]

print("🧪 Testing Query Decomposition...\n")

In [ ]:
# Query decomposition prompt for LLM
decomposition_prompt = """You are a query decomposition specialist.

Your task: Break down complex analytical questions into simple, answerable sub-questions.

Rules:
1. Each sub-question should be independently answerable
2. Sub-questions should be ordered logically (dependencies first)
3. Cover all aspects of the original question
4. Keep sub-questions simple and specific
5. Number each sub-question clearly

Output format:
**Sub-Question 1:** [question]
**Sub-Question 2:** [question]
...

Complex Question: {question}

Break this down:"""

for i, question in enumerate(complex_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"COMPLEX QUESTION {i}")
    print("=" * 80)
    print(f"\n📝 Original: {question}\n")
    
    # Get decomposition
    prompt = decomposition_prompt.format(question=question)
    response = llm.invoke([HumanMessage(content=prompt)])
    
    print("🔍 Decomposition:")
    print(response.content)
    print("\n" + "-" * 80)

### ✍️ Section 4 Observations

**Questions to answer:**
1. Are the decompositions logical?
2. Do sub-questions cover all aspects of the original question?
3. Are any sub-questions too complex or unclear?
4. Would you change the decomposition approach?

**Your notes:**
- 
- 

---

## Section 5: Query Planning

**Purpose:** Test how schema analysis results are converted into Genie queries.

**What to test:**
- Are queries specific and clear?
- Do queries use correct column names?
- Are filters applied correctly?
- Are calculations explained properly?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 5: QUERY PLANNING")
print("=" * 80)
print()

In [ ]:
# Create query planner agent
from src.agent_simple import create_query_planner_agent

query_planner = create_query_planner_agent(llm)

print("✓ Query Planner Agent created")

In [ ]:
# Test query planning with analyzed questions
test_scenarios = [
    {
        "question": "Show me sentiment for bangalore",
        "analysis": """**ANSWERABLE: YES**

**SELECTED COLUMNS AND TABLES:**
Table: main.default.pc_sales
Required columns:
  - sentiment_score: Customer sentiment metric
  - city: Location filter

Optional/context columns:
  - date: Temporal context
  - product_name: Product context"""
    },
    {
        "question": "Calculate conversion rate for Q4 2024",
        "analysis": """**ANSWERABLE: YES**

**SELECTED COLUMNS AND TABLES:**
Table: main.default.web_analytics
Required columns:
  - conversion_count: Number of conversions
  - visitor_count: Total visitors
  - date: Time filter

**BUSINESS LOGIC APPLIED:**
Conversion Rate = (conversion_count / visitor_count) * 100"""
    }
]

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n{'=' * 80}")
    print(f"SCENARIO {i}")
    print("=" * 80)
    print(f"\n📝 Question: {scenario['question']}")
    print(f"\n📊 Schema Analysis:\n{scenario['analysis']}")
    
    # Create test state
    test_state = {
        "messages": [
            HumanMessage(content=scenario['question']),
            AIMessage(content=f"Schema Analysis:\n{scenario['analysis']}")
        ],
        "schema_info": schema_info,
        "rag_context": "",
        "iterations": 0
    }
    
    # Run query planner
    result = query_planner(test_state)
    
    # Display results
    print(f"\n🎯 Generated Query:\n")
    formatted_query = result.get('formatted_query', 'No query generated')
    print(formatted_query)
    
    if result.get('messages'):
        for msg in result['messages']:
            if isinstance(msg, AIMessage) and "Query Plan" in msg.content:
                print(f"\n💬 Planner Response:\n")
                print(msg.content)
    
    print("\n" + "-" * 80)

### ✍️ Section 5 Observations

**Questions to answer:**
1. Are the generated queries clear and specific?
2. Do queries use the correct column names from schema?
3. Are filters formatted properly?
4. Are calculated metrics explained correctly?

**Your notes:**
- 
- 

---

## Section 6: Genie Execution

**Purpose:** Test actual query execution via Databricks Genie.

**⚠️ Warning:** This section executes real queries against Databricks.

**What to test:**
- Does Genie understand the queries?
- Are results returned correctly?
- Are results relevant to the question?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 6: GENIE EXECUTION")
print("=" * 80)
print()

In [ ]:
# Create Genie executor
from src.agent_simple import create_genie_executor_node
from src.services.smart_cache import SmartSQLCache

# Initialize cache if enabled
sql_cache = None
if config.cache.enabled:
    from src.utils.embeddings import get_embedding_service
    embeddings_service = get_embedding_service()
    sql_cache = SmartSQLCache(
        embedding_service=embeddings_service,
        ttl_seconds=config.cache.ttl_seconds,
        similarity_threshold=config.cache.similarity_threshold,
        max_entries=config.cache.max_entries
    )
    print("✓ Smart Cache enabled")
else:
    print("⚠️  Cache disabled")

genie_executor = create_genie_executor_node(workspace_client, sql_cache)

print("✓ Genie Executor created")

In [ ]:
# Test queries (simple examples - modify as needed)
test_genie_queries = [
    "Show me the first 5 rows from the pc_sales table",
    "Count the total number of records in pc_sales"
]

print("\n⚠️  WARNING: This will execute real queries against Databricks!")
print("\nTest queries:")
for i, q in enumerate(test_genie_queries, 1):
    print(f"  {i}. {q}")

# Uncomment to run tests
# run_genie_tests = input("\nRun Genie tests? (yes/no): ").strip().lower()
run_genie_tests = "no"  # Change to "yes" to enable

In [ ]:
if run_genie_tests == "yes":
    for i, query in enumerate(test_genie_queries, 1):
        print(f"\n{'=' * 80}")
        print(f"GENIE TEST {i}")
        print("=" * 80)
        print(f"\n📝 Query: {query}\n")
        
        # Create test state
        test_state = {
            "formatted_query": query,
            "iterations": 0,
            "messages": []
        }
        
        try:
            # Execute
            result = genie_executor(test_state)
            
            # Display results
            print("✓ Execution completed\n")
            print(f"Cache Hit: {result.get('cache_hit', False)}")
            print(f"\n📊 Results:\n")
            
            if result.get('messages'):
                for msg in result['messages']:
                    if isinstance(msg, AIMessage):
                        print(msg.content)
        
        except Exception as e:
            print(f"❌ Error: {e}")
        
        print("\n" + "-" * 80)
else:
    print("\n⏭️  Skipping Genie execution tests")
    print("   Set run_genie_tests = 'yes' to enable")

### ✍️ Section 6 Observations

**Questions to answer:**
1. Did Genie execute the queries successfully?
2. Were the results relevant and accurate?
3. Were there any errors or unexpected behaviors?
4. Did cache work correctly (if enabled)?

**Your notes:**
- 
- 

---

## Section 7: Result Validation

**Purpose:** Validate that Genie's responses match the original question intent.

**What to test:**
- Does the result answer the question?
- Are filters applied correctly (e.g., city = bangalore)?
- Are calculated values reasonable?
- Are there any obvious errors or mismatches?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 7: RESULT VALIDATION")
print("=" * 80)
print()

In [ ]:
# Result validation prompt
validation_prompt = """You are a result validation specialist.

Your task: Verify that Genie's response correctly answers the user's question.

Check for:
1. **Relevance:** Does the result relate to the question?
2. **Filters:** Are requested filters (city, date, etc.) applied?
3. **Completeness:** Does it answer all aspects of the question?
4. **Accuracy:** Are values reasonable and calculations correct?
5. **Errors:** Are there any error messages or empty results?

Original Question: {question}

Genie Result: {result}

Validation:
**VALID:** YES/NO

**Reasoning:**
[Explain your validation decision]

**Issues Found (if any):**
- [list any problems]

**Suggested Follow-up Questions (if needed):**
- [questions to get missing information]
"""

print("✓ Result validation template ready")

In [ ]:
# Test validation with sample scenarios
validation_scenarios = [
    {
        "question": "Show me sentiment for bangalore",
        "result": """Results:
| city      | sentiment_score |
|-----------|----------------|
| bangalore | 0.85           |
| bangalore | 0.72           |
| bangalore | 0.91           |"""
    },
    {
        "question": "Show me sentiment for bangalore",
        "result": """Results:
| city      | sentiment_score |
|-----------|----------------|
| delhi     | 0.65           |
| mumbai    | 0.78           |
| bangalore | 0.85           |"""
    },
    {
        "question": "Calculate conversion rate for Q4 2024",
        "result": """Results:
| metric           | value |
|------------------|-------|
| conversion_rate  | 3.2%  |"""
    }
]

for i, scenario in enumerate(validation_scenarios, 1):
    print(f"\n{'=' * 80}")
    print(f"VALIDATION TEST {i}")
    print("=" * 80)
    print(f"\n📝 Question: {scenario['question']}")
    print(f"\n📊 Result:\n{scenario['result']}")
    
    # Validate
    prompt = validation_prompt.format(
        question=scenario['question'],
        result=scenario['result']
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    
    print(f"\n✅ Validation:")
    print(response.content)
    print("\n" + "-" * 80)

### ✍️ Section 7 Observations

**Questions to answer:**
1. Did the validation catch incorrect results?
2. Were the reasoning explanations clear?
3. Were suggested follow-ups helpful?
4. What validation rules would you add?

**Your notes:**
- 
- 

---

## Section 8: Follow-up Generation

**Purpose:** Generate intelligent follow-up questions based on results.

**What to test:**
- Are follow-ups relevant and insightful?
- Do they help with deeper analysis?
- Are they answerable with available data?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 8: FOLLOW-UP GENERATION")
print("=" * 80)
print()

In [ ]:
# Follow-up generation prompt
followup_prompt = """You are an analytical insights specialist.

Your task: Generate intelligent follow-up questions to deepen the analysis.

Original Question: {question}

Result Summary: {result}

Available Tables and Columns:
{schema_info}

Generate 3-5 analytical follow-up questions that:
1. Dig deeper into the results
2. Compare with other metrics or dimensions
3. Identify trends or patterns
4. Are answerable with available data
5. Provide business insights

Output format:
**Follow-up 1:** [question]
**Rationale:** [why this follow-up is valuable]

**Follow-up 2:** [question]
**Rationale:** [why this follow-up is valuable]

...
"""

print("✓ Follow-up generation template ready")

In [ ]:
# Test follow-up generation
followup_scenarios = [
    {
        "question": "Show me sentiment for bangalore",
        "result": "Average sentiment score for bangalore: 0.82 (across 150 records)"
    },
    {
        "question": "Calculate conversion rate for Q4 2024",
        "result": "Q4 2024 conversion rate: 3.2% (1,200 conversions out of 37,500 visitors)"
    }
]

for i, scenario in enumerate(followup_scenarios, 1):
    print(f"\n{'=' * 80}")
    print(f"FOLLOW-UP TEST {i}")
    print("=" * 80)
    print(f"\n📝 Original Question: {scenario['question']}")
    print(f"\n📊 Result: {scenario['result']}")
    
    # Generate follow-ups
    prompt = followup_prompt.format(
        question=scenario['question'],
        result=scenario['result'],
        schema_info=schema_info[:1000]  # Truncate for brevity
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    
    print(f"\n🔍 Generated Follow-ups:")
    print(response.content)
    print("\n" + "-" * 80)

### ✍️ Section 8 Observations

**Questions to answer:**
1. Are the follow-ups relevant and insightful?
2. Do they help uncover deeper insights?
3. Are they answerable with available data?
4. Would you use these follow-ups in practice?

**Your notes:**
- 
- 

---

## Section 9: End-to-End Test

**Purpose:** Test the complete system with a real question.

**What to test:**
- Does the entire flow work correctly?
- Are all agents coordinating properly?
- Is the final answer accurate and complete?

In [ ]:
print("\n" + "=" * 80)
print("SECTION 9: END-TO-END TEST")
print("=" * 80)
print()

In [ ]:
# Get the full agent
from src.agent_simple import get_agent
import uuid

agent = get_agent()
thread_id = str(uuid.uuid4())

print("✓ Full agent initialized")
print(f"✓ Thread ID: {thread_id}")

In [ ]:
# Test question
test_question = "Show me sentiment for bangalore"  # Modify as needed

print(f"\n📝 Question: {test_question}\n")
print("🤖 Processing...\n")
print("=" * 80)

In [ ]:
# Create input state
input_state = {
    "messages": [HumanMessage(content=test_question)],
    "next_agent": "",
    "iterations": 0,
    "rag_checked": False,
    "schema_analyzed": False,
    "query_planned": False,
    "genie_executed": False,
    "original_question": "",
    "schema_info": "",
    "formatted_query": "",
    "final_answer": "",
    "rag_context": "",
    "rag_answer": "",
    "rag_can_answer": False,
    "rag_similarity": 0.0,
    "cache_checked": False,
    "cache_hit": False,
    "cached_result": "",
    "is_answerable": False,
    "needs_clarification": False,
    "clarification_provided": False,
}

# Run agent
try:
    result = agent.invoke(
        input_state,
        config={"configurable": {"thread_id": thread_id}}
    )
    
    print("\n✅ EXECUTION COMPLETED\n")
    print("=" * 80)
    
    # Show final answer
    final_answer = result.get("final_answer", "")
    if not final_answer and result.get("messages"):
        final_answer = result["messages"][-1].content
    
    print("\n📊 FINAL ANSWER:\n")
    print(final_answer)
    print("\n" + "=" * 80)
    
    # Show execution details
    print("\n⚙️  EXECUTION DETAILS:\n")
    print(f"  Iterations: {result.get('iterations', 0)}")
    print(f"  Answerable: {result.get('is_answerable', 'Unknown')}")
    print(f"  RAG Used: {result.get('rag_checked', False)}")
    print(f"  Cache Hit: {result.get('cache_hit', False)}")
    print(f"  Genie Executed: {result.get('genie_executed', False)}")
    
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

### ✍️ Section 9 Observations

**Questions to answer:**
1. Did the complete flow work as expected?
2. Was the final answer accurate and complete?
3. Were there any unexpected behaviors?
4. What improvements would you make?

**Your notes:**
- 
- 

---

## Summary & Next Steps

### Key Findings

After running all sections, summarize your observations:

1. **Schema Understanding:**
   - 

2. **Column Selection:**
   - 

3. **RAG Integration:**
   - 

4. **Query Decomposition:**
   - 

5. **Query Planning:**
   - 

6. **Genie Execution:**
   - 

7. **Result Validation:**
   - 

8. **Follow-up Generation:**
   - 

### Recommended Improvements

1. 
2. 
3. 

### Next Actions

- [ ] Add column descriptions to Unity Catalog tables
- [ ] Create RAG documents for business logic
- [ ] Adjust LLM prompts based on findings
- [ ] Test with more complex queries
- [ ] Implement result validation in main code
- [ ] Add follow-up question generation
- [ ] Test query decomposition integration